In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

os.makedirs("models", exist_ok=True)

df = pd.read_csv("outputs/model_dataset.csv")

targets = ["avg_d_mbps", "avg_u_mbps", "avg_lat_ms"]

feature_cols = [
    "latitude",
    "longitude",
    "year",
    "quarter",
    "block_number",
    "city",
    "area",
    "typeOfArea",
    "region",
    "digital_elevation_model",
    "tests",
    "nearest_tower_distance_km",
    "tower_count_1km",
    "tower_count_2km",
    "tower_count_5km",
    "visible_tower_count_5km",
    "unique_bands_5km",
]

In [2]:
feature_cols += [c for c in df.columns if c.startswith("tower_count_5km_")]
feature_cols += [c for c in df.columns if c.startswith("rat_count_5km_")]
feature_cols += [c for c in df.columns if c.startswith("ratsubtype_count_5km_")]

feature_cols = list(dict.fromkeys([c for c in feature_cols if c in df.columns]))

X = df[feature_cols].copy()
Y = df[targets].copy()

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [3]:
train_mask = df["year"] < 2025
test_mask = df["year"] == 2025

if test_mask.sum() == 0:
    train_mask = np.arange(len(df)) % 5 != 0
    test_mask = ~train_mask

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
Y_train = Y.loc[train_mask]
Y_test = Y.loc[test_mask]

def eval_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }

models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    ),
}

results = {}
saved_models = {}

for model_name, estimator in models.items():
    results[model_name] = {}
    saved_models[model_name] = {}

    for target in targets:
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", estimator)
        ])

        pipe.fit(X_train, Y_train[target])
        preds = pipe.predict(X_test)

        results[model_name][target] = eval_metrics(Y_test[target], preds)
        saved_models[model_name][target] = pipe

In [6]:
metrics_df = []
for model_name, target_dict in results.items():
    for target, metric_dict in target_dict.items():
        row = {"model": model_name, "target": target}
        row.update(metric_dict)
        metrics_df.append(row)

metrics_df = pd.DataFrame(metrics_df)
metrics_df.to_csv("outputs/model_metrics.csv", index=False)

In [7]:
reference_points = df[[
    "latitude", "longitude", "block_number", "city", "area",
    "typeOfArea", "region", "digital_elevation_model", "tests"
]].drop_duplicates().reset_index(drop=True)

towers = pd.read_csv("outputs/towers_clean.csv")

bundle = {
    "models": saved_models,
    "metrics": results,
    "feature_columns": feature_cols,
    "reference_points": reference_points,
    "towers": towers
}

joblib.dump(bundle, "models/model_bundle.joblib")

print("Saved:")
print("- models/model_bundle.joblib")
print("- outputs/model_metrics.csv")
print("\nMetrics:")
print(metrics_df)

Saved:
- models/model_bundle.joblib
- outputs/model_metrics.csv

Metrics:
               model      target         MAE        RMSE        R2
0  linear_regression  avg_d_mbps  132.427285  178.641906  0.063828
1  linear_regression  avg_u_mbps   15.883178   22.240857  0.026861
2  linear_regression  avg_lat_ms   14.284729   36.554446 -0.037063
3      random_forest  avg_d_mbps  129.666856  184.213818  0.004518
4      random_forest  avg_u_mbps   15.313697   22.375195  0.015070
5      random_forest  avg_lat_ms   15.350793   43.784244 -0.487855
